# Agentic RAG - ArXiv Paper Curator

### Interactive demonstration of the adaptive retrieval pipeline

In [7]:
import sys
import os
from pathlib import Path
import requests
import time

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# Find project root
current_dir = Path.cwd()
if current_dir.name == "week7" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent.parent

if project_root.exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("⚠ Project root not found - check directory structure")

# Load .env file if it exists
env_file = project_root / ".env"
if env_file.exists():
    print(f"\n✓ Loading environment from: {env_file}")
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                if key not in os.environ:
                    os.environ[key] = value
    print("✓ Environment variables loaded")
else:
    print(f"\n⚠ No .env file found at: {env_file}")
    print("  Run: cp .env.example .env")
    print("  Then add your JINA_API_KEY, LANGFUSE_PUBLIC_KEY, and LANGFUSE_SECRET_KEY")

# Configuration for notebook tests
REQUEST_TIMEOUT = 300
TRUNCATE_ANSWERS = True
TRUNCATE_LENGTH = 200

print("\n✓ Setup complete")

Python Version: 3.12.13
Project root: /home/ashwin/arxiv-paper-curator

✓ Loading environment from: /home/ashwin/arxiv-paper-curator/.env
✓ Environment variables loaded

✓ Setup complete


In [ ]:
services = {
    "FastAPI": "http://localhost:8000/api/v1/health",
    "Ollama": "http://localhost:11434/api/version"
}

all_healthy = True
for service_name, url in services.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"✓ {service_name}: Healthy")
        else:
            print(f"✗ {service_name}: HTTP {response.status_code}")
            all_healthy = False
    except:
        print(f"✗ {service_name}: Not accessible")
        all_healthy = False

# Check if Ollama model is available
print("\nChecking Ollama model availability...")
try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    if response.status_code == 200:
        models = [m['name'] for m in response.json().get('models', [])]
        if 'llama3.2:1b' in models:
            print("✓ llama3.2:1b model is available")
        else:
            print("⚠ llama3.2:1b not found. Run: docker exec rag-ollama ollama pull llama3.2:1b")
            all_healthy = False
except:
    print("⚠ Could not check Ollama models")

if all_healthy:
    print("\n✓ All services ready for Week 7!")
else:
    print("\n⚠ Some services need attention. Run: docker compose up --build -d")

WEEK 7 SERVICE HEALTH CHECK
✓ FastAPI: Healthy
✓ Ollama: Healthy

Checking Ollama model availability...
✓ llama3.2:1b model is available

✓ All services ready for Week 7!


## 1. Test Traditional RAG (Baseline)

First, let's test the traditional RAG endpoint to establish a baseline.

In [12]:
print("TRADITIONAL RAG TEST (Baseline)")
print("=" * 40)

question = "What are attention mechanisms?"
print(f"Question: {question}\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": "llama3.2:1b"
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Traditional RAG ({elapsed:.1f}s)")
        
        # Display answer with configurable truncation
        answer = data['answer']
        if TRUNCATE_ANSWERS and len(answer) > TRUNCATE_LENGTH:
            print(f"\nAnswer: {answer[:TRUNCATE_LENGTH]}...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else:
            print(f"\nAnswer: {answer}")
        
        # Display sources with validation
        sources = data.get('sources', [])
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources[:3], 1):  # Show first 3
                if isinstance(source, dict):
                    print(f"  {i}. {source.get('title', 'Unknown')}")
                else:
                    print(f"  {i}. {source}")
        
        print(f"Search mode: {data.get('search_mode', 'unknown')}")
    else:
        print(f"✗ Request failed: {response.status_code}")
        
except Exception as e:
    print(f"✗ Error: {e}")

TRADITIONAL RAG TEST (Baseline)
Question: What are attention mechanisms?

✓ Traditional RAG (134.7s)

Answer: Attention mechanisms play a crucial role in understanding sequential data, such as text or speech. Unlike traditional recurrent neural networks (RNNs), which only process adjacent elements in time, at...
(truncated, full length: 1365 chars)

Sources: 3 papers
  1. https://arxiv.org/pdf/2608.13463.pdf
  2. https://arxiv.org/pdf/2608.13547.pdf
  3. https://arxiv.org/pdf/2608.13492.pdf
Search mode: hybrid


## 2. Test Agentic RAG - Scenario 1: Out-of-Scope Rejection

Test if the guardrail correctly rejects queries outside the ML/NLP domain.

In [15]:
print("AGENTIC RAG - SCENARIO 1: Out-of-Scope Rejection")
print("=" * 50)

question = "What is a dog?"
print(f"Question: {question}")
print("Expected: Guardrail should reject (score < 60) and explain scope\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        print(f"\nAnswer: {data['answer']}")
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")
        
        # Check if guardrail score is in reasoning steps
        guardrail_step = next(
            (s for s in data.get('reasoning_steps', []) if 'validated' in s.lower() and 'score' in s.lower()),
            None
        )
        if guardrail_step:
            print(f"\nGuardrail validation: {guardrail_step}")
        
        if data.get('retrieval_attempts', 0) == 0:
            print("\n✓ SUCCESS: Query correctly rejected by guardrail (no retrieval)!")
        else:
            print("\n⚠ UNEXPECTED: Query should have been rejected without retrieval")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")

AGENTIC RAG - SCENARIO 1: Out-of-Scope Rejection
Question: What is a dog?
Expected: Guardrail should reject (score < 60) and explain scope

✓ Agentic RAG (0.2s)

Answer: I apologize, but I can only help with questions about academic research papers in Computer Science, Artificial Intelligence, and Machine Learning from arXiv.

Your question: 'What is a dog?'

This appears to be outside my domain of expertise. For questions like this, you might want to try:
- General-purpose AI assistants for broad knowledge questions
- Domain-specific resources for topics outside CS/AI/ML
- Technical documentation if asking about specific software/tools

If you have a question about AI/ML research papers, I'd be happy to help!

Retrieval attempts: 0

Reasoning steps:
  1. Validated query scope (score: 50/100)
  2. Generated answer from context

Guardrail validation: Validated query scope (score: 50/100)

✓ SUCCESS: Query correctly rejected by guardrail (no retrieval)!


## 3. Test Agentic RAG - Scenario 2: Successful Retrieval

Test if the agent correctly retrieves and grades documents for research questions.

In [27]:
print("AGENTIC RAG - SCENARIO 2: Successful Retrieval")
print("=" * 50)

question = "What are transformers in machine learning?"
print(f"Question: {question}")
print("Expected: Agent should pass guardrail, retrieve documents and generate answer\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": "llama3.2:3b"
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        
        # Display answer with better formatting
        answer = data.get('answer', '')
        print(f"\nAnswer:\n{'-'*50}")
        if TRUNCATE_ANSWERS and len(answer) > 500:  # Use longer limit for detailed answers
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else:
            print(answer)
        print('-'*50)
        
        # Display sources with validation
        sources = data.get('sources', [])
        print(f"\nSources: {len(sources)} papers")
        if sources:
            for i, source in enumerate(sources, 1):
                if isinstance(source, dict):
                    print(f"  {i}. {source.get('title', source.get('id', 'Unknown'))}")
                elif isinstance(source, str):
                    print(f"  {i}. {source}")
                else:
                    print(f"  {i}. {str(source)}")

        print(f"Guardrail score: {data.get('guardrail_score')}/100")
                            
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")
        

        # Check rewritten_query field
        if data.get('rewritten_query') is None:
            print("\n✓ Query was not rewritten (worked on first attempt)")
        else:
            print(f"\n→ Query was rewritten to: {data['rewritten_query']}")
        
        if data.get('retrieval_attempts', 0) >= 1:
            print("\n✓ SUCCESS: Agent retrieved and used documents!")
        else:
            print("\n⚠ UNEXPECTED: Agent didn't retrieve for research question")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")

AGENTIC RAG - SCENARIO 2: Successful Retrieval
Question: What are transformers in machine learning?
Expected: Agent should pass guardrail, retrieve documents and generate answer

✓ Agentic RAG (215.6s)

Answer:
--------------------------------------------------
Transformers are a type of neural network architecture that have gained significant attention in recent years for their ability to model complex relationships between input data. In machine learning, a transformer is essentially an encoder-decoder model with self-attention mechanisms.

One of the key characteristics of transformers is that they use a novel approach to modeling sequential data, such as text or images. The core idea behind this is to decompose the input sequence into individual el...
(truncated, full length: 2948 chars)
--------------------------------------------------

Sources: 0 papers
Guardrail score: None/100

Retrieval attempts: 1

Reasoning steps:
  1. Validated query scope (score: 64/100)
  2. Retrieved do

## 4. Test Agentic RAG - Scenario 3: Query Rewriting

Test if the agent rewrites vague queries for better results.

In [28]:
print("AGENTIC RAG - SCENARIO 3: Query Rewriting")
print("=" * 50)

question = "Tell me about ML stuff"
print(f"Question: {question}")
print("Expected: Agent may rewrite query if documents aren't relevant\n")

start_time = time.time()

try:
    response = requests.post(
        "http://localhost:8000/api/v1/ask-agentic",
        json={
            "query": question,
            "top_k": 3,
            "use_hybrid": True,
            "model": "llama3.2:3b"
        },
        timeout=REQUEST_TIMEOUT
    )
    
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Agentic RAG ({elapsed:.1f}s)")
        
        # Display answer with better formatting
        answer = data.get('answer', '')
        print(f"\nAnswer:\n{'-'*50}")
        if TRUNCATE_ANSWERS and len(answer) > 500:
            print(answer[:500] + "...")
            print(f"(truncated, full length: {len(answer)} chars)")
        else:
            print(answer)
        print('-'*50)
        
        print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
        print(f"\nReasoning steps:")
        for i, step in enumerate(data.get('reasoning_steps', []), 1):
            print(f"  {i}. {step}")
        
        # Check for guardrail validation step
        print("\nValidating guardrail and rewrite steps:")
        reasoning_steps = data.get('reasoning_steps', [])
        if any("validated" in step.lower() for step in reasoning_steps):
            guardrail_step = next(s for s in reasoning_steps if "validated" in s.lower())
            print(f"  ✓ Guardrail validation: {guardrail_step}")
        else:
            print("  ⚠ Guardrail validation step missing")
        
        # Check for query rewriting
        if data.get('rewritten_query'):
            print(f"\n✓ Query was rewritten!")
            print(f"  Original: {question}")
            print(f"  Rewritten: {data['rewritten_query']}")
        elif data.get('retrieval_attempts', 0) > 1:
            print("\n→ Multiple retrieval attempts detected")
            if any("rewritten" in step.lower() for step in reasoning_steps):
                print("  ✓ Rewrite step found in reasoning")
            else:
                print("  ⚠ Multiple attempts but no rewrite info")
        else:
            print("\n→ Query worked on first attempt (no rewrite needed)")
        
        if data.get('retrieval_attempts', 0) > 1:
            print(f"\n✓ Agent performed {data['retrieval_attempts']} retrieval attempts")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error: {e}")

AGENTIC RAG - SCENARIO 3: Query Rewriting
Question: Tell me about ML stuff
Expected: Agent may rewrite query if documents aren't relevant

✓ Agentic RAG (160.9s)

Answer:
--------------------------------------------------
The user's question is about ML-related topics. I'll provide a comprehensive and accurate answer based solely on the retrieved papers.

Modern image classification models excel when trained on single-task-specific datasets but often struggle to generalize across domains and difficulty levels, as highlighted by Perkins et al. (2020) in their paper "MLLM-Routed Heterogeneous Ensembles for Robust Cross-Dataset Image Classification" [1]. This challenges the traditional approach of training models on...
(truncated, full length: 3003 chars)
--------------------------------------------------

Retrieval attempts: 1

Reasoning steps:
  1. Validated query scope (score: 60/100)
  2. Retrieved documents (1 attempt(s))
  3. Graded documents (1 relevant)
  4. Generated answer from 

In [29]:
print("AGENTIC RAG - SCENARIO 4: Multiple Out-of-Scope Queries")
print("=" * 50)

test_queries = [
    ("What is a dog?", "Biology question"),
    ("What's the weather today?", "Weather question"),
    ("Hello, how are you?", "Greeting"),
]

print("Testing guardrail rejection with various non-ML/NLP queries:\n")

for query, description in test_queries:
    print(f"Query: {query}")
    print(f"Type: {description}")
    
    try:
        response = requests.post(
            "http://localhost:8000/api/v1/ask-agentic",
            json={"query": query, "top_k": 3, "use_hybrid": True},
            timeout=30
        )
        
        if response.status_code == 200:
            data = response.json()
            
            # Check if rejected (no retrieval)
            is_rejected = data['retrieval_attempts'] == 0
            
            # Get guardrail score from reasoning if available
            guardrail_step = next(
                (s for s in data['reasoning_steps'] if 'validated' in s.lower() and 'score' in s.lower()),
                None
            )
            
            print(f"Result: {'✓ REJECTED' if is_rejected else '✗ ACCEPTED'} (attempts: {data['retrieval_attempts']})")
            if guardrail_step:
                print(f"Guardrail: {guardrail_step}")
        else:
            print(f"✗ Request failed: {response.status_code}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print("-" * 50)

AGENTIC RAG - SCENARIO 4: Multiple Out-of-Scope Queries
Testing guardrail rejection with various non-ML/NLP queries:

Query: What is a dog?
Type: Biology question
✗ Error: HTTPConnectionPool(host='localhost', port=8000): Read timed out. (read timeout=30)
--------------------------------------------------
Query: What's the weather today?
Type: Weather question
Result: ✓ REJECTED (attempts: 0)
Guardrail: Validated query scope (score: 59/100)
--------------------------------------------------
Query: Hello, how are you?
Type: Greeting
✗ Error: HTTPConnectionPool(host='localhost', port=8000): Read timed out. (read timeout=30)
--------------------------------------------------


## 5. Interactive Testing

Try your own questions!

In [ ]:
def ask_agentic(question: str, show_full_answer: bool = False):
    """Helper function to test agentic RAG.
    
    Args:
        question: The question to ask
        show_full_answer: If True, show full answer regardless of TRUNCATE_ANSWERS setting
    """
    print(f"Question: {question}\n")
    
    start = time.time()
    
    try:
        response = requests.post(
            "http://localhost:8000/api/v1/ask-agentic",
            json={"query": question, "top_k": 3, "use_hybrid": True},
            timeout=REQUEST_TIMEOUT
        )
        
        elapsed = time.time() - start
        
        if response.status_code == 200:
            data = response.json()
            print(f"✓ Response in {elapsed:.1f}s\n")
            
            # Display answer
            answer = data.get('answer', '')
            print(f"Answer:\n{'-'*50}")
            if not show_full_answer and TRUNCATE_ANSWERS and len(answer) > 500:
                print(answer[:500] + "...")
                print(f"(truncated, full length: {len(answer)} chars)")
            else:
                print(answer)
            print('-'*50)
            
            # Display metadata
            print(f"\nRetrieval attempts: {data.get('retrieval_attempts', 0)}")
            
            # Display sources with validation
            sources = data.get('sources', [])
            print(f"Sources: {len(sources)}")
            if sources:
                for i, source in enumerate(sources[:3], 1):  # Show first 3
                    if isinstance(source, dict):
                        print(f"  {i}. {source.get('title', source.get('id', 'Unknown'))}")
                    elif isinstance(source, str):
                        print(f"  {i}. {source}")
            
            # Display reasoning
            print(f"\nReasoning:")
            for step in data.get('reasoning_steps', []):
                print(f"  • {step}")
        else:
            print(f"✗ Error: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"✗ Exception: {e}")

# Try it!
ask_agentic("How does BERT differ from GPT?")

In [ ]:
# Try more questions
ask_agentic("What is the capital of France?")  # Should reject as out-of-scope